# Sprint 6 – Notebook 2: Numerical Feature Engineering


In [1]:
import pandas as pd
import numpy as np

# Reusable synthetic customer-shopping dataset (no CSV required)
df = pd.DataFrame({
    "Customer_ID": [101, 102, 103, 104, 105, 106],
    "Age": [23, 31, 45, 28, 39, 52],
    "Annual_Income": [300000, 550000, 900000, 420000, 720000, 1100000],
    "Product_Price": [1200, 2500, 5000, 1800, 3500, 6000],
    "Quantity": [2, 1, 3, 4, 2, 1],
    "Discount_Percent": [10, 0, 15, 5, 20, 10],
    "Previous_Spending": [1800, 1500, 11000, 5000, 4500, 4000],
    "Days_Since_Last_Purchase": [10, 40, 5, 22, 12, 65],
    "Previous_Order_Count": [3, 2, 8, 5, 4, 1],
    "Gender": ["F", "M", "F", "M", "F", "M"],
    "City": ["Hassan", "Mysore", "Bengaluru", "Hassan", "Mysore", "Bengaluru"],
    "Membership_Type": ["Silver", "Gold", "Gold", "Silver", "Gold", "Silver"],
    "Product_Category": ["Clothing", "Electronics", "Furniture", "Clothing", "Electronics", "Furniture"],
    "Purchase_Date": pd.to_datetime(["2026-01-05", "2026-01-10", "2026-02-12", "2026-02-20", "2026-03-04", "2026-03-15"]),
    "Payment_Method": ["UPI", "Card", "UPI", "Cash", "Card", "UPI"],
    "Repeat_Purchase": [1, 0, 1, 1, 1, 0]
})
df["Total_Spending"] = df["Product_Price"] * df["Quantity"] * (1 - df["Discount_Percent"] / 100)
print(df[["Customer_ID", "Annual_Income", "Product_Price", "Quantity", "Total_Spending"]])

   Customer_ID  Annual_Income  Product_Price  Quantity  Total_Spending
0          101         300000           1200         2          2160.0
1          102         550000           2500         1          2500.0
2          103         900000           5000         3         12750.0
3          104         420000           1800         4          6840.0
4          105         720000           3500         2          5600.0
5          106        1100000           6000         1          5400.0


## 1. Mathematical Transformations

### Definition
Mathematical transformations create numerical features by applying arithmetic or mathematical functions to existing columns. They can reveal relationships that raw values do not show. The transformation should match the meaning of the data. Always check for missing, negative, or zero values before applying a function.

### Business Example
A retailer combines product price and quantity to understand order size. Price alone does not tell the value of a purchase. A transformed order-value feature provides a more useful business signal. It can be compared across different customers.

### AI/ML Use
Models often learn more easily from meaningful derived features. Transformations can improve signal strength and reduce problematic scales. They should be computed without using future information. Validate whether the new feature actually helps on held-out data.

### Python Example

In [3]:
df["Gross_Order_Value"] = df["Product_Price"] * df["Quantity"]
print(df[["Product_Price", "Quantity", "Gross_Order_Value"]].head())

   Product_Price  Quantity  Gross_Order_Value
0           1200         2               2400
1           2500         1               2500
2           5000         3              15000
3           1800         4               7200
4           3500         2               7000


### Explanation
The multiplication combines two original numerical columns. The resulting feature represents order value before discount. Each row is calculated independently. It may be useful for predicting future customer behavior.

**When and why to use:** Use this technique when its derived quantity has a clear business meaning and can be calculated from information available at prediction time. Compare it with the original features on validation data before keeping it.

## 2. Addition

### Definition
Addition combines two compatible amounts into one total. It is useful when several parts contribute to a single business quantity. Both values should use compatible units. Adding unrelated features may create misleading numbers.

### Business Example
A business may combine spending from the current order with earlier spending. The result describes spending accumulated up to that point. This can support customer segmentation. It should only include transactions already known at prediction time.

### AI/ML Use
Total spending can summarize purchasing activity in one feature. It may help models distinguish frequent and occasional buyers. Addition is simple and interpretable. Beware of adding future transactions when predicting earlier outcomes.

### Python Example

In [5]:
df["Known_Spending_Total"] = df["Previous_Spending"] + df["Total_Spending"]
print(df[["Previous_Spending", "Total_Spending", "Known_Spending_Total"]].head())

   Previous_Spending  Total_Spending  Known_Spending_Total
0               1800          2160.0                3960.0
1               1500          2500.0                4000.0
2              11000         12750.0               23750.0
3               5000          6840.0               11840.0
4               4500          5600.0               10100.0


### Explanation
The code adds historical spending and the current transaction value. It produces a combined spending amount for each customer. The values remain in the same currency. Use this only when both amounts are known at prediction time.

**When and why to use:** Use this technique when its derived quantity has a clear business meaning and can be calculated from information available at prediction time. Compare it with the original features on validation data before keeping it.

## 3. Subtraction

### Definition
Subtraction measures how much one numerical value exceeds another. It is useful for gaps, balances, and changes. The order of subtraction affects the sign. Interpret positive and negative results carefully.

### Business Example
A retailer compares the listed order value with the discounted amount paid. Their difference represents the discount saving. Larger savings may influence purchasing decisions. The feature can help describe promotion effects.

### AI/ML Use
A difference feature can expose useful gaps between related measurements. It can sometimes be easier to interpret than two separate columns. The model may learn patterns linked to those gaps. Confirm the feature is available before the prediction is made.

### Python Example

In [7]:
df["Discount_Saving"] = df["Gross_Order_Value"] - df["Total_Spending"]
print(df[["Gross_Order_Value", "Total_Spending", "Discount_Saving"]].head())

   Gross_Order_Value  Total_Spending  Discount_Saving
0               2400          2160.0            240.0
1               2500          2500.0              0.0
2              15000         12750.0           2250.0
3               7200          6840.0            360.0
4               7000          5600.0           1400.0


### Explanation
The code subtracts paid spending from the original order value. The result is the amount saved through discount. Positive values indicate savings. The feature can help evaluate promotion-related behavior.

**When and why to use:** Use this technique when its derived quantity has a clear business meaning and can be calculated from information available at prediction time. Compare it with the original features on validation data before keeping it.

## 4. Multiplication

### Definition
Multiplication combines numerical quantities to represent a joint effect. It is useful for prices times units and other interactions. It can create large values, so check scale. Only multiply features when the result has a meaningful interpretation.

### Business Example
A shop sells products at a unit price and in varying quantities. Multiplying them gives the value before discount. This is more informative than quantity alone. The same idea works for hours times hourly pay.

### AI/ML Use
Interaction features may capture effects that models do not learn directly. They are especially useful for simpler linear models. A meaningful product can improve predictive power. Unnecessary interactions can increase noise.

### Python Example

In [9]:
df["Gross_Order_Value"] = df["Product_Price"] * df["Quantity"]
print(df[["Product_Price", "Quantity", "Gross_Order_Value"]].head())

   Product_Price  Quantity  Gross_Order_Value
0           1200         2               2400
1           2500         1               2500
2           5000         3              15000
3           1800         4               7200
4           3500         2               7000


### Explanation
Each product price is multiplied by its quantity. The new column shows the order amount before discount. It is a simple numerical interaction. Its unit is currency.

**When and why to use:** Use this technique when its derived quantity has a clear business meaning and can be calculated from information available at prediction time. Compare it with the original features on validation data before keeping it.

## 5. Division

### Definition
Division expresses one quantity relative to another. It is useful when raw amounts differ greatly across customers. The denominator must not be zero. Ratios from division need clear units and interpretation.

### Business Example
A retailer divides spending by the number of units bought. This gives the average amount paid per unit. It allows fairer comparisons between small and large orders. A zero quantity must be handled safely.

### AI/ML Use
Per-unit measures can help models compare customers on a common basis. They may reveal purchasing patterns hidden in totals. Division can also create unstable values for very small denominators. Check the resulting distribution.

### Python Example

In [11]:
df["Paid_Per_Unit"] = df["Total_Spending"] / df["Quantity"].replace(0, np.nan)
print(df[["Total_Spending", "Quantity", "Paid_Per_Unit"]].head())

   Total_Spending  Quantity  Paid_Per_Unit
0          2160.0         2         1080.0
1          2500.0         1         2500.0
2         12750.0         3         4250.0
3          6840.0         4         1710.0
4          5600.0         2         2800.0


### Explanation
The code divides each paid order total by its unit count. Zero quantities are replaced with missing values to avoid division by zero. The result represents paid price per unit. Missing results should be handled appropriately.

**When and why to use:** Use this technique when its derived quantity has a clear business meaning and can be calculated from information available at prediction time. Compare it with the original features on validation data before keeping it.

## 6. Ratios

### Definition
A ratio compares the size of one numerical quantity with another. Ratios are useful when absolute values alone are hard to compare. They should have a meaningful numerator and denominator. Very small denominators can create extreme ratios.

### Business Example
A customer spending ₹5,000 and earning ₹3,00,000 differs from one spending the same amount with a lower income. A spending-to-income ratio provides context. It expresses spending relative to financial capacity. Annual and transaction periods must be interpreted carefully.

### AI/ML Use
Ratios can improve comparisons across customer groups. They may reveal relative behavior more clearly than raw spending. Ratios are often useful in financial and retail models. Check for leakage and denominator problems.

### Python Example

In [13]:
df["Order_Income_Ratio"] = df["Total_Spending"] / df["Annual_Income"].replace(0, np.nan)
print(df[["Total_Spending", "Annual_Income", "Order_Income_Ratio"]].head())

   Total_Spending  Annual_Income  Order_Income_Ratio
0          2160.0         300000            0.007200
1          2500.0         550000            0.004545
2         12750.0         900000            0.014167
3          6840.0         420000            0.016286
4          5600.0         720000            0.007778


### Explanation
The ratio divides current order spending by annual income. A larger value means the order is larger relative to income. It is a dimensionless comparison, but the time periods differ. Interpret it as order size relative to annual income, not a monthly spending rate.

**When and why to use:** Use this technique when its derived quantity has a clear business meaning and can be calculated from information available at prediction time. Compare it with the original features on validation data before keeping it.

## 7. Percentages

### Definition
A percentage expresses a share or change out of 100. It makes proportions easier to communicate. The denominator must be meaningful and nonzero. Percentages can exceed 100 for some types of change.

### Business Example
Retailers use discount percentages to communicate price reductions. A 20% discount means 20 of every 100 currency units is removed. Percentages make promotions comparable across different prices. They are familiar to business teams.

### AI/ML Use
Percentage features can standardize proportions across observations. Models may use them to learn promotion effects or customer behavior. Percentages should be consistently represented as 0–100 or 0–1. Avoid mixing the two scales accidentally.

### Python Example

In [15]:
df["Calculated_Discount_Pct"] = 100 * df["Discount_Saving"] / df["Gross_Order_Value"].replace(0, np.nan)
print(df[["Discount_Percent", "Calculated_Discount_Pct"]].head())

   Discount_Percent  Calculated_Discount_Pct
0                10                     10.0
1                 0                      0.0
2                15                     15.0
3                 5                      5.0
4                20                     20.0


### Explanation
The code calculates savings as a percentage of original order value. Multiplying by 100 converts a fraction into percent. The calculated result matches the given discount rate. The denominator is protected against zero.

**When and why to use:** Use this technique when its derived quantity has a clear business meaning and can be calculated from information available at prediction time. Compare it with the original features on validation data before keeping it.

## 8. Differences

### Definition
A difference feature captures the gap between two related measurements. It may represent improvement, decline, or distance. The sign indicates direction when subtraction is used. Differences are most useful when the two values are comparable.

### Business Example
A store compares the current purchase with a previous spending amount. A positive difference means the current purchase is larger. A negative difference means it is smaller. This can help describe changes in purchasing activity.

### AI/ML Use
Difference features may help a model detect changes in customer behavior. They are often more informative than isolated measurements. Comparisons must use consistent definitions and periods. Avoid comparing future observations with historical ones.

### Python Example

In [17]:
df["Spending_Difference"] = df["Total_Spending"] - df["Previous_Spending"]
print(df[["Total_Spending", "Previous_Spending", "Spending_Difference"]].head())

   Total_Spending  Previous_Spending  Spending_Difference
0          2160.0               1800                360.0
1          2500.0               1500               1000.0
2         12750.0              11000               1750.0
3          6840.0               5000               1840.0
4          5600.0               4500               1100.0


### Explanation
Current order spending is compared with the stored previous spending amount. Positive and negative results show direction of change. The values are measured in currency. This is a difference between two amounts, not necessarily a time-series growth rate.

**When and why to use:** Use this technique when its derived quantity has a clear business meaning and can be calculated from information available at prediction time. Compare it with the original features on validation data before keeping it.

## 9. Aggregations

### Definition
Aggregations summarize multiple values using operations such as sum, mean, minimum, and maximum. They can be computed within a group of records. Aggregated features often capture broader customer or product behavior. Be careful not to include records from the future.

### Business Example
A retailer calculates average order value for each product category. This summarizes purchasing patterns for clothing, furniture, and electronics. Managers can compare categories more easily. The same technique can summarize customer-level transactions.

### AI/ML Use
Group aggregates provide context beyond individual records. They can be predictive for recommendation and demand models. To avoid leakage, compute learned aggregates using training data only or strictly past records. Validate how unseen groups are handled.

### Python Example

In [19]:
category_mean = df.groupby("Product_Category")["Total_Spending"].transform("mean")
df["Category_Mean_Spending"] = category_mean
print(df[["Product_Category", "Total_Spending", "Category_Mean_Spending"]].head())

  Product_Category  Total_Spending  Category_Mean_Spending
0         Clothing          2160.0                  4500.0
1      Electronics          2500.0                  4050.0
2        Furniture         12750.0                  9075.0
3         Clothing          6840.0                  4500.0
4      Electronics          5600.0                  4050.0


### Explanation
The code groups records by product category and calculates mean spending. transform() returns a value aligned to every original row. Customers in the same category share the category mean. For real prediction, build aggregates using training or historical data only.

**When and why to use:** Use this technique when its derived quantity has a clear business meaning and can be calculated from information available at prediction time. Compare it with the original features on validation data before keeping it.

## 10. Log Transformation

### Definition
A log transformation compresses large positive values more than small ones. It can reduce right skew in numerical data. log1p(x) calculates the log of one plus x and supports zero values. Negative values below −1 are not valid for log1p.

### Business Example
Order amounts may contain a few very expensive purchases. These large values can dominate a graph or model. A log scale makes the spread easier to examine. It preserves the ordering of nonnegative values.

### AI/ML Use
Log features can help some models handle heavily skewed quantities. They can make multiplicative patterns easier to learn. The transformation does not automatically improve every algorithm. Compare results before and after on validation data.

### Python Example

In [21]:
df["Log_Spending"] = np.log1p(df["Total_Spending"])
print(df[["Total_Spending", "Log_Spending"]].head())

   Total_Spending  Log_Spending
0          2160.0      7.678326
1          2500.0      7.824446
2         12750.0      9.453365
3          6840.0      8.830689
4          5600.0      8.630700


### Explanation
np.log1p() applies a logarithm after adding one. It reduces the numerical gap between very large and small spending values. Zero would be transformed safely to zero. This dataset has nonnegative spending values.

**When and why to use:** Use it when the feature distribution or model requirements justify the transformation; compare performance and check for invalid inputs.

## 11. Square Root Transformation

### Definition
Square root transformation reduces the spread of nonnegative values. It is gentler than a logarithmic transformation. It is often considered for counts or moderately skewed data. Negative inputs are not supported in real-valued square roots.

### Business Example
Customers may purchase one, two, or many items. Taking the square root reduces the influence of larger counts. The transformed values still increase as quantity increases. This can make a count feature less uneven.

### AI/ML Use
Square root features can reduce skew in count-based inputs. They may help algorithms sensitive to feature scale or distribution. The transformation should match the data type. Compare predictive performance before adopting it.

### Python Example

In [23]:
df["Sqrt_Quantity"] = np.sqrt(df["Quantity"])
print(df[["Quantity", "Sqrt_Quantity"]].head())

   Quantity  Sqrt_Quantity
0         2       1.414214
1         1       1.000000
2         3       1.732051
3         4       2.000000
4         2       1.414214


### Explanation
The code applies square root to each quantity. Large quantities are compressed relative to small ones. The transformed feature stays nonnegative. It is easy to calculate and interpret.

**When and why to use:** Use it when the feature distribution or model requirements justify the transformation; compare performance and check for invalid inputs.

## 12. Power Transformation

### Definition
Power transformations raise values to a chosen exponent. Squaring emphasizes larger values, while fractional powers compress them. The effect depends on the chosen power. Very large values may become even more extreme after squaring.

### Business Example
A retailer might test whether high quantities have a stronger-than-linear relationship with sales outcomes. Squared quantity represents this possibility. It can highlight high-volume purchases. Domain knowledge should guide whether the feature makes sense.

### AI/ML Use
Polynomial features help linear models learn nonlinear relationships. They can also increase feature size and overfitting risk. Use them selectively and validate performance. Tree models may already learn nonlinear effects without explicit powers.

### Python Example

In [25]:
df["Quantity_Squared"] = df["Quantity"] ** 2
print(df[["Quantity", "Quantity_Squared"]].head())

   Quantity  Quantity_Squared
0         2                 4
1         1                 1
2         3                 9
3         4                16
4         2                 4


### Explanation
The code squares each order quantity. A quantity of four becomes sixteen. This emphasizes larger purchases more strongly. Use the new feature only when it improves the model.

**When and why to use:** Use it when the feature distribution or model requirements justify the transformation; compare performance and check for invalid inputs.

## 13. Absolute Difference

### Definition
Absolute difference measures the size of a gap without its direction. It is always zero or positive. This is useful when the magnitude of change matters more than increase or decrease. It should not replace signed difference when direction matters.

### Business Example
A store measures how far the current purchase differs from earlier spending. Both a large increase and a large decrease represent major changes. Absolute difference captures the size of that change. It does not show which way the change occurred.

### AI/ML Use
Absolute gaps can help detect unusual changes in behavior. They are useful when both positive and negative deviations matter. Models may benefit from both signed and absolute differences. Avoid creating redundant features without checking value.

### Python Example

In [27]:
df["Absolute_Spending_Gap"] = (df["Total_Spending"] - df["Previous_Spending"]).abs()
print(df[["Spending_Difference", "Absolute_Spending_Gap"]].head())

   Spending_Difference  Absolute_Spending_Gap
0                360.0                  360.0
1               1000.0                 1000.0
2               1750.0                 1750.0
3               1840.0                 1840.0
4               1100.0                 1100.0


### Explanation
The code subtracts previous spending from current spending. abs() removes the negative sign if present. The resulting feature measures only gap size. It cannot indicate whether spending rose or fell.

**When and why to use:** Use this technique when its derived quantity has a clear business meaning and can be calculated from information available at prediction time. Compare it with the original features on validation data before keeping it.

## 14. Relative Difference

### Definition
Relative difference expresses a change compared with a reference value. It allows comparisons across different starting amounts. The reference value must be clearly chosen. Zero or near-zero reference values need special handling.

### Business Example
A ₹500 increase is significant for a customer previously spending ₹1,000 but smaller for one spending ₹20,000. Relative difference captures that context. It compares the change with the earlier amount. This supports more meaningful comparisons.

### AI/ML Use
Relative changes can be more predictive than raw changes when customers have different spending levels. They can also be noisy when baseline amounts are small. Consider clipping extreme ratios when justified. Use historical reference values only.

### Python Example

In [29]:
df["Relative_Spending_Change"] = (df["Total_Spending"] - df["Previous_Spending"]) / df["Previous_Spending"].replace(0, np.nan)
print(df[["Previous_Spending", "Total_Spending", "Relative_Spending_Change"]].head())

   Previous_Spending  Total_Spending  Relative_Spending_Change
0               1800          2160.0                  0.200000
1               1500          2500.0                  0.666667
2              11000         12750.0                  0.159091
3               5000          6840.0                  0.368000
4               4500          5600.0                  0.244444


### Explanation
The difference is divided by previous spending. A value of 0.5 means a 50% increase relative to the previous amount. Negative values indicate a decrease. Zero denominators are converted to missing values.

**When and why to use:** Use this technique when its derived quantity has a clear business meaning and can be calculated from information available at prediction time. Compare it with the original features on validation data before keeping it.

## 15. Rate Features

### Definition
A rate expresses an amount or event count per unit of time or exposure. Rates allow comparisons across different durations. The unit must be clearly stated. A rate is different from a raw count.

### Business Example
A retailer may compare previous orders with the time since the last purchase. A simple orders-per-day proxy gives an activity measure. However, the days since last purchase is not the full observation window. A true purchase rate requires the actual historical period length.

### AI/ML Use
Rates can summarize activity intensity in customer models. They are useful when customers have different observation periods. Ensure time windows are measured consistently. Do not interpret an approximate proxy as a verified rate.

### Python Example

In [31]:
df["Orders_Per_Day_Proxy"] = df["Previous_Order_Count"] / df["Days_Since_Last_Purchase"].replace(0, np.nan)
print(df[["Previous_Order_Count", "Days_Since_Last_Purchase", "Orders_Per_Day_Proxy"]].head())

   Previous_Order_Count  Days_Since_Last_Purchase  Orders_Per_Day_Proxy
0                     3                        10              0.300000
1                     2                        40              0.050000
2                     8                         5              1.600000
3                     5                        22              0.227273
4                     4                        12              0.333333


### Explanation
The code divides previous order count by days since the last purchase. It creates a simple numerical proxy for activity. Because the denominator is not the full order-history duration, it is not a true order rate. For a real model, use orders divided by the measured observation window.

**When and why to use:** Use this technique when its derived quantity has a clear business meaning and can be calculated from information available at prediction time. Compare it with the original features on validation data before keeping it.

## 16. Normalized Features

### Definition
Normalized features rescale values to a common range, often 0 to 1. This prevents large numerical units from dominating some algorithms. Min-max normalization depends on the observed minimum and maximum. New values may fall outside the original range.

### Business Example
Income may be measured in lakhs while quantity is a small integer. A common scale makes these variables easier to compare. It can also improve some distance-based calculations. Extreme income values may compress most normalized observations.

### AI/ML Use
Normalization is useful for KNN, neural networks, and other scale-sensitive methods. Fit the scaler only on training data to avoid leakage. Apply the fitted scaler to validation and test data. Tree-based models usually do not require it.

### Python Example

In [33]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size=0.33, random_state=42)
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train[["Annual_Income"]])
test_scaled = scaler.transform(test[["Annual_Income"]])
print("Train normalized:", train_scaled.round(2).ravel())
print("Test normalized:", test_scaled.round(2).ravel())

Train normalized: [1.   0.71 0.44 0.  ]
Test normalized: [-0.18  0.19]


### Explanation
MinMaxScaler learns the minimum and maximum from training data only. fit_transform() scales the training values. transform() applies the same learned scale to test values. This avoids using test statistics during preprocessing.

**When and why to use:** Use it when the feature distribution or model requirements justify the transformation; compare performance and check for invalid inputs.